In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import datetime

from memory.short_term_memory import ShortTermMemory
from memory.long_term_memory import LongTermMemory

from tools.facts_retrieve import retrieve_facts_schema
from function_call import select_service
from prompts import SYSTEM_PROMPT

load_dotenv()
client = OpenAI(
    base_url="https://freellmapi-seyc.onrender.com/v1",
    api_key=os.environ.get("FREE_LLM_API")
)


stm = ShortTermMemory(mode ="summary_buffer", llm_client=client)
ltm = LongTermMemory(llm_client=client)


session_id = ltm.session_id

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
user_input = input("Input your query: ")
stm.add(assistant_msg={"role": "user", "content": user_input}, role="tool")
messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat
while True:
    if user_input.strip().lower() == "exit":
        ltm.add(stm.slide_chat)
        break

    print("\n\n"+str(messages[1:]))
    response = client.chat.completions.create(
        model="auto",
        messages=messages,
        tools=[retrieve_facts_schema],
        tool_choice='auto'
    )
    msg = response.choices[0].message

    print(response.choices[0].message)


    if msg.tool_calls:
        assistant_tool_msg = {
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        }
        stm.add(assistant_msg=assistant_tool_msg, role="tool")

        for tool_call in msg.tool_calls:
            tool_result = select_service(tool_call.function)
            stm.add(
                assistant_msg={
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                },
                role="tool",
            )
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat

    if response.choices[0].finish_reason == "stop":
        stm.add(assistant_msg={"role": "assistant", "content": msg.content}, role="tool")
        user_input = input("Input your query: ")
        stm.add(assistant_msg={"role": "user", "content": user_input}, role="tool")
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat + [{"role": "user", "content": user_input}]



[{'role': 'user', 'content': 'Hi'}]
ChatCompletionMessage(content='Hello! How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user just says "Hi". This is a greeting. No need to retrieve memory. We should respond friendly.')


[{'role': 'assistant', 'content': 'Hello! How can I help you today?'}, {'role': 'user', 'content': 'Hi'}, {'role': 'user', 'content': 'How are you?'}]
ChatCompletionMessage(content='I’m doing great, thank you! How can I assist you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User greets. No need memory. Simple response.')


[{'role': 'assistant', 'content': 'Hello! How can I help you today?'}, {'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'I’m doing great, thank you! How can I assist you today?'}, {'role': 'user', 'content': 'How are you?'}, {'role': 'user', 'content': 'What is 

In [21]:
stm.slide_chat

[{'role': 'assistant', 'content': 'Hello! How can I help you today?'},
 {'role': 'user', 'content': 'Hi'},
 {'role': 'assistant',
  'content': 'I’m doing great, thank you! How can I assist you today?'},
 {'role': 'user', 'content': 'How are you?'},
 {'role': 'assistant',
  'content': 'I’m just a virtual assistant, so I don’t have feelings, but I’m here and ready to help you! You can call me\u202fChatGPT. How can I assist you today?'},
 {'role': 'user', 'content': 'What is your name?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'fc_ae4c1254-9e35-47fe-acf4-b259253f4b2a',
    'function': {'arguments': '{"query":"user name","top_k":5}',
     'name': 'retrieve_facts'},
    'type': 'function'}]},
 {'role': 'tool',
  'tool_call_id': 'fc_ae4c1254-9e35-47fe-acf4-b259253f4b2a',
  'content': '["The user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma", "user\'s name is Satyam Sharma"]'},
 {'role': 